# Spotify Intelligence System
**End-to-End Data Science & Machine Learning Project**

---
**Phases Covered:**
1. Data Cleaning & Preprocessing
2. Exploratory Data Analysis (EDA)
3. Hit Song Prediction (ML Classification)
4. Music Recommendation System
5. Song Clustering
6. Streamlit App (see `app/streamlit_app.py`)

**Dataset:** [Spotify Tracks Dataset – Kaggle](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)

## 0. Setup

In [ ]:
# Install dependencies (run once)
# !pip install -r ../requirements.txt -q

In [ ]:
import sys
from pathlib import Path

# Add project root to path so `src` is importable from notebooks/
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

# src modules
from src.preprocessing import (
    load_raw_data, clean_data, engineer_features,
    scale_features, build_rec_matrix, build_cluster_matrix,
    save_cleaned_data, save_artefacts,
    AUDIO_FEATURES, REC_FEATURES, CLUSTER_FEATURES,
)
from src.visualization import (
    plot_correlation_heatmap,
    plot_popularity_distribution,
    plot_top_genres,
    plot_audio_feature_distributions,
    plot_feature_correlation_heatmap,
    plot_model_comparison,
    plot_feature_importance,
)
from src.prediction import (
    prepare_hit_data, apply_smote,
    get_default_models, train_and_evaluate,
    get_classification_report, get_confusion_matrix,
    get_feature_importance, predict_with_threshold,
    train_balanced_rf, save_model, predict_hit_probability,
)
from src.clustering import (
    plot_elbow, fit_kmeans, plot_pca_clusters, save_kmeans,
)
from src.recommenders import (
    normal_recommendation, artist_songs, recommend_genres,
    mood_based, cluster_recommendation,
    hybrid_recommendation, compare_all_recommenders,
)

print("All modules imported successfully!")

## 1. Data Loading & Preprocessing

In [ ]:
# Load raw data
df_raw = load_raw_data("../data/dataset.csv")
df_raw.head()

In [ ]:
# Inspect nulls and duplicates before cleaning
print("Nulls per column:")
print(df_raw.isnull().sum())
print(f"\nDuplicated rows: {df_raw.duplicated().sum():,}")

In [ ]:
# Clean: drop nulls + duplicates
df_clean = clean_data(df_raw)

In [ ]:
# EDA: raw correlation heatmap
plot_correlation_heatmap(df_clean, title="Raw Feature Correlation")

In [ ]:
# Feature engineering
df, label_encoder = engineer_features(df_clean)
df.head()

In [ ]:
# Scale audio features (creates df_scaled; df stays unscaled for recommenders)
df_scaled, scaler = scale_features(df, AUDIO_FEATURES)

In [ ]:
# Persist cleaned data and preprocessing artefacts
save_cleaned_data(df, path="../data/cleaned_spotify.csv")
save_artefacts(scaler, AUDIO_FEATURES, models_dir="../models")

## 2. Exploratory Data Analysis

In [ ]:
# Popularity distribution
plot_popularity_distribution(df)

In [ ]:
# Top 20 genres by average popularity (interactive Plotly chart)
plot_top_genres(df, n=20)

In [ ]:
# Distribution of each audio feature
plot_audio_feature_distributions(df, AUDIO_FEATURES)

In [ ]:
# Correlation of audio features with popularity
plot_feature_correlation_heatmap(df, AUDIO_FEATURES)

## 3. Hit Song Prediction

In [ ]:
# Prepare train/test split
X_train, X_test, y_train, y_test = prepare_hit_data(df, AUDIO_FEATURES)

In [ ]:
# Resample minority class
X_resampled, y_resampled = apply_smote(X_train, y_train)

### 3.1 Benchmark all models

In [ ]:
models = get_default_models()
results_df, fitted_models = train_and_evaluate(models, X_resampled, X_test, y_resampled, y_test)
results_df

In [ ]:
# Visual comparison
plot_model_comparison(results_df, metric="ROC AUC")

### 3.2 Inspect best model (Random Forest)

In [ ]:
best_model = fitted_models["Random Forest"]
get_classification_report(best_model, X_test, y_test, "Random Forest")

In [ ]:
# Confusion matrix
get_confusion_matrix(best_model, X_test, y_test)

In [ ]:
# Feature importances
importance_df = get_feature_importance(best_model, AUDIO_FEATURES)
plot_feature_importance(importance_df)
importance_df

### 3.3 Threshold tuning & balanced RF

In [ ]:
# Custom threshold (0.3) to improve hit recall
predict_with_threshold(best_model, X_test, y_test, threshold=0.3)

In [ ]:
# Balanced Random Forest (uses class_weight='balanced' instead of SMOTE)
rf_balanced, _ = train_balanced_rf(X_train, X_test, y_train, y_test)

### 3.4 SHAP explainability (optional — uncomment to run)
Requires `shap` to be installed.

In [ ]:
# import shap
# explainer = shap.TreeExplainer(best_model)
# X_sample = X_test.sample(500, random_state=42)
# shap_values = explainer.shap_values(X_sample)
# shap.summary_plot(shap_values, X_sample, feature_names=AUDIO_FEATURES)

### 3.5 Save model

In [ ]:
save_model(best_model, path="../models/hit_predictor.pkl")

## 4. Music Recommendation System

In [ ]:
# Build recommendation feature matrix (uses unscaled df)
df_rec, X_rec, rec_scaler = build_rec_matrix(df, REC_FEATURES)

### 4.1 Content-based (cosine similarity)

In [ ]:
normal_recommendation("Du Hast", df_rec, X_rec, top_n=10)

### 4.2 Artist top songs

In [ ]:
artist_songs("michael jackson", df_rec, top_n=10)

### 4.3 Genre top songs

In [ ]:
recommend_genres("pop", df_rec, top_n=20)

### 4.4 Mood-based playlist

In [ ]:
mood_based("party", df, top_n=10)

## 5. Song Clustering

In [ ]:
# Build cluster feature matrix (sampled subset)
df_cluster, X_cluster, cluster_scaler = build_cluster_matrix(df, n_sample=20_000)

### 5.1 Elbow method — choose optimal K

In [ ]:
plot_elbow(X_cluster, k_range=range(1, 11))

### 5.2 Fit K-Means (K=5)

In [ ]:
df_cluster, kmeans = fit_kmeans(df_cluster, X_cluster, n_clusters=5)
df_cluster.head()

### 5.3 PCA visualisation

In [ ]:
pca, X_pca = plot_pca_clusters(df_cluster, X_cluster)

In [ ]:
# Save K-Means model
save_kmeans(kmeans, path="../models/kmeans.pkl")

## 6. Cluster-Based & Hybrid Recommendations

In [ ]:
cluster_recommendation("Thriller", df_cluster, top_n=10)

In [ ]:
hybrid_recommendation("Thriller", df_rec, X_rec, df_cluster, top_n=10)

In [ ]:
# Side-by-side comparison of all three engines
compare_all_recommenders("Dream On", df_rec, X_rec, df_cluster, top_n=5)

## 7. Hit Prediction Inference Demo

In [ ]:
# Predict hit probability for a hypothetical new track
new_song = {
    "danceability": 0.82,
    "energy": 0.88,
    "loudness": -5,
    "speechiness": 0.06,
    "acousticness": 0.12,
    "instrumentalness": 0.00,
    "liveness": 0.18,
    "valence": 0.79,
    "tempo": 126,
    "duration_mins": 3.1,
}
probability = predict_hit_probability(
    new_song,
    model_path="../models/hit_predictor.pkl",
    scaler_path="../models/scaler.pkl",
    feature_columns_path="../models/feature_columns.pkl",
)
print(f"Predicted hit probability: {probability:.2%}")